# Vaani KWS — V2 Dataset Preparation
---
**Project:** SIH 2026 — Low Latency & Efficient Voice Activator for Edge Devices  
**Keyword:** "Vaani"  
**Version:** V2  

## V2 Motivation
V1 achieved ~97.71% accuracy at threshold 0.90, but suffered catastrophic silence false-fires:
- 23/30 false activations on a ~6.93s quiet-room recording
- Average P(Vaani) = 0.871 on silence  
- Max P(Vaani) = 0.9985 on silence  

**V2 Goal:** Explicitly include silence/room-tone data and preserve negative diversity.

### What This Notebook Does
1. Discovers and validates all source audio
2. Performs stratified sampling of Speech Commands (~100/category)
3. Splits into train / validation / test (70 / 15 / 15)
4. Per-speaker 70/15/15 for positive data (every speaker in every split)
5. Source-aware splitting for silence and background (no cross-split leakage)
6. Copies selected files into `model_v2/data/`
7. Builds a reproducible manifest with SHA-256 hashes
8. Runs comprehensive leakage verification
9. Prints final dataset report

### What This Notebook Does NOT Do
- No augmentation (handled in notebook 02)
- No feature extraction / MFCC / log-Mel (handled in notebook 02)
- No neural-network training
- No TensorFlow imports
- No modification of original data under `dataset/raw/`

## 1. Configuration

In [21]:
# ============================================================
# SECTION 1 — CONFIGURATION
# ============================================================

import os
import sys
import json
import wave
import shutil
import hashlib
import re
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Audio requirements ───────────────────────────────────────
EXPECTED_SR = 16000
EXPECTED_CHANNELS = 1
EXPECTED_SAMPLE_WIDTH = 2   # PCM16 = 2 bytes per sample
EXPECTED_DURATION_S = 1.0
DURATION_TOLERANCE_S = 0.05

# ── Split ratios ─────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Speech Commands sampling ────────────────────────────────
TARGET_PER_SC_CATEGORY = 100

# ── Labels ───────────────────────────────────────────────────
LABEL_POSITIVE = 1   # "vaani"
LABEL_NEGATIVE = 0   # everything else

SPLITS = ["train", "validation", "test"]
CATEGORIES = [
    "positive",
    "negative_silence",
    "negative_background",
    "negative_speech_commands",
]

print("Configuration loaded.")
print(f"  SEED = {SEED}")
print(f"  Split ratios: train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")
print(f"  Target Speech Commands per category: {TARGET_PER_SC_CATEGORY}")

Configuration loaded.
  SEED = 42
  Split ratios: train=0.7, val=0.15, test=0.15
  Target Speech Commands per category: 100


## 2. Locate Project Root

In [22]:
# ============================================================
# SECTION 2 — PROJECT ROOT
# ============================================================

def find_project_root():
    """Walk up from CWD to find the project root
    (directory containing both 'dataset/' and 'model_v1/').
    """
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "dataset").is_dir() and (candidate / "model_v1").is_dir():
            return candidate
    # Hardcoded fallback
    fallback = Path(r"c:/Users/Mayank Singh/Codes/SIH 2026")
    if fallback.is_dir():
        return fallback
    raise FileNotFoundError(
        "Could not find PROJECT_ROOT. "
        "Expected a directory containing 'dataset/' and 'model_v1/'."
    )

PROJECT_ROOT = find_project_root()
DATASET_RAW  = PROJECT_ROOT / "dataset" / "raw"
MODEL_V2     = PROJECT_ROOT / "model_v2"
TEMP_DIR     = PROJECT_ROOT / "temp"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_RAW:  {DATASET_RAW}")
print(f"MODEL_V2:     {MODEL_V2}")

# ── Source directories ───────────────────────────────────────
POSITIVE_DIR = DATASET_RAW / "positive"
assert POSITIVE_DIR.is_dir(), f"Missing: {POSITIVE_DIR}"

# Silence — check multiple possible locations
_silence_candidates = [
    DATASET_RAW / "negative_silence",
    DATASET_RAW / "silence" / "processed",
    DATASET_RAW / "silence",
]
SILENCE_DIR = None
for _sc in _silence_candidates:
    if _sc.is_dir() and list(_sc.glob("*.wav")):
        SILENCE_DIR = _sc
        break
if SILENCE_DIR is None:
    raise FileNotFoundError(
        f"Could not find silence WAV directory. Checked: {_silence_candidates}"
    )

BACKGROUND_DIR = DATASET_RAW / "negative_background"
assert BACKGROUND_DIR.is_dir(), f"Missing: {BACKGROUND_DIR}"

SPEECH_CMD_DIR = DATASET_RAW / "negative_speech_commands"
assert SPEECH_CMD_DIR.is_dir(), f"Missing: {SPEECH_CMD_DIR}"

print()
print(f"Positive dir:        {POSITIVE_DIR}")
print(f"Silence dir:         {SILENCE_DIR}")
print(f"Background dir:      {BACKGROUND_DIR}")
print(f"Speech commands dir: {SPEECH_CMD_DIR}")

PROJECT_ROOT: c:\Users\Mayank Singh\Codes\SIH 2026
DATASET_RAW:  c:\Users\Mayank Singh\Codes\SIH 2026\dataset\raw
MODEL_V2:     c:\Users\Mayank Singh\Codes\SIH 2026\model_v2

Positive dir:        c:\Users\Mayank Singh\Codes\SIH 2026\dataset\raw\positive
Silence dir:         c:\Users\Mayank Singh\Codes\SIH 2026\dataset\raw\silence\processed
Background dir:      c:\Users\Mayank Singh\Codes\SIH 2026\dataset\raw\negative_background
Speech commands dir: c:\Users\Mayank Singh\Codes\SIH 2026\dataset\raw\negative_speech_commands


In [23]:
# ── Create V2 directory structure ─────────────────────────────

V2_DIRS = {
    "data":       MODEL_V2 / "data",
    "features":   MODEL_V2 / "features",
    "checkpoints": MODEL_V2 / "checkpoints",
    "evaluation": MODEL_V2 / "evaluation",
    "exports":    MODEL_V2 / "exports",
    "scripts":    MODEL_V2 / "scripts",
    "manifests":  MODEL_V2 / "data" / "manifests",
}

for name, path in V2_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    for cat in CATEGORIES:
        (V2_DIRS["data"] / split / cat).mkdir(parents=True, exist_ok=True)

# Temp directory for any intermediate files
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("V2 directory structure created.")

V2 directory structure created.


## 3. Source Dataset Inspection

In [24]:
# ============================================================
# SECTION 3 — DISCOVER SOURCE FILES
# ============================================================

positive_files    = sorted(POSITIVE_DIR.glob("*.wav"))
silence_files     = sorted(SILENCE_DIR.glob("*.wav"))
background_files  = sorted(BACKGROUND_DIR.glob("*.wav"))
speech_cmd_files  = sorted(SPEECH_CMD_DIR.glob("*.wav"))

print("=" * 60)
print("SOURCE FILE COUNTS")
print("=" * 60)
print(f"  Positive (Vaani):         {len(positive_files):>7,}")
print(f"  Negative silence:         {len(silence_files):>7,}")
print(f"  Negative background:      {len(background_files):>7,}")
print(f"  Negative speech commands: {len(speech_cmd_files):>7,}")
print(f"  {'─' * 44}")
total_source = (len(positive_files) + len(silence_files) +
                len(background_files) + len(speech_cmd_files))
print(f"  Total source files:       {total_source:>7,}")

# ── Positive speakers ────────────────────────────────────────
def extract_positive_speaker(filepath):
    """Positive filenames: <speaker>_<number>.wav  →  speaker name."""
    parts = filepath.stem.split('_')
    # Last part is numeric index; everything before is speaker name
    if len(parts) > 1 and parts[-1].isdigit():
        return '_'.join(parts[:-1])
    return "unknown"

pos_speakers = defaultdict(list)
for f in positive_files:
    spk = extract_positive_speaker(f)
    pos_speakers[spk].append(f)

print()
print("Positive speakers:")
for spk in sorted(pos_speakers):
    print(f"  {spk:<12s}: {len(pos_speakers[spk]):>4d} recordings")

# ── Silence source recordings ────────────────────────────────
def extract_silence_source(filepath):
    """Silence filenames: Recording (53)_clip_00000.wav
    Source = everything before '_clip_'.
    """
    m = re.match(r'^(.+?)_clip_', filepath.name)
    if m:
        return m.group(1)
    return filepath.stem  # fallback: entire stem

silence_sources = defaultdict(list)
for f in silence_files:
    src = extract_silence_source(f)
    silence_sources[src].append(f)

print()
print("Silence source recordings:")
for src in sorted(silence_sources):
    print(f"  {src!r}: {len(silence_sources[src]):>4d} clips")

# ── Background source recordings ─────────────────────────────
def extract_background_source(filepath):
    """Background filenames: background_doing_the_dishes_0000.wav
    Source = everything before the last '_NNNN' numeric suffix.
    """
    name = filepath.stem
    # Remove trailing _NNNN
    m = re.match(r'^(.+)_\d{4,}$', name)
    if m:
        return m.group(1)
    return name

bg_sources = defaultdict(list)
for f in background_files:
    src = extract_background_source(f)
    bg_sources[src].append(f)

print()
print("Background source recordings:")
for src in sorted(bg_sources):
    print(f"  {src}: {len(bg_sources[src]):>4d} clips")

SOURCE FILE COUNTS
  Positive (Vaani):             704
  Negative silence:             307
  Negative background:          791
  Negative speech commands: 105,829
  ────────────────────────────────────────────
  Total source files:       107,631

Positive speakers:
  ananya      :   78 recordings
  ark         :   98 recordings
  ishita      :   78 recordings
  mayank      :   96 recordings
  umang       :  100 recordings
  vitthal     :  254 recordings

Silence source recordings:
  'Recording (53)':  307 clips

Background source recordings:
  background_doing_the_dishes:  189 clips
  background_dude_miaowing:  122 clips
  background_exercise_bike:  121 clips
  background_pink_noise:  119 clips
  background_running_tap:  121 clips
  background_white_noise:  119 clips


## 4. Audio Validation

Validate that all source files meet requirements:
- Decodable WAV
- 16 kHz sample rate
- Mono (1 channel)
- PCM16 (2-byte samples)
- ~1 second duration (±50ms tolerance)

Files that fail are **reported but not silently deleted**.

In [25]:
# ============================================================
# SECTION 4 — AUDIO VALIDATION
# ============================================================

def validate_wav(filepath):
    """Validate a single WAV file. Returns dict with properties and issues."""
    result = {
        "filepath": str(filepath),
        "filename": filepath.name,
        "readable": False,
        "sample_rate": None,
        "channels": None,
        "sample_width": None,
        "duration_s": None,
        "issues": [],
    }
    
    try:
        with wave.open(str(filepath), 'rb') as wf:
            result["sample_rate"]   = wf.getframerate()
            result["channels"]      = wf.getnchannels()
            result["sample_width"]  = wf.getsampwidth()
            result["duration_s"]    = wf.getnframes() / wf.getframerate()
            result["readable"]      = True
    except Exception as e:
        result["issues"].append(f"Cannot read WAV header: {e}")
        return result
    
    if result["sample_rate"] != EXPECTED_SR:
        result["issues"].append(f"SR={result['sample_rate']} (expected {EXPECTED_SR})")
    if result["channels"] != EXPECTED_CHANNELS:
        result["issues"].append(f"CH={result['channels']} (expected {EXPECTED_CHANNELS})")
    if result["sample_width"] != EXPECTED_SAMPLE_WIDTH:
        result["issues"].append(f"SW={result['sample_width']} (expected {EXPECTED_SAMPLE_WIDTH})")
    if result["duration_s"] is not None:
        if abs(result["duration_s"] - EXPECTED_DURATION_S) > DURATION_TOLERANCE_S:
            result["issues"].append(
                f"Duration={result['duration_s']:.3f}s (expected {EXPECTED_DURATION_S}s)"
            )
    return result

def validate_file_set(files, category_name):
    """Validate a list of WAV files."""
    results = []
    for f in tqdm(files, desc=f"Validating {category_name}", leave=True):
        r = validate_wav(f)
        r["category"] = category_name
        results.append(r)
    
    issues_count = sum(1 for r in results if r["issues"])
    unreadable   = sum(1 for r in results if not r["readable"])
    print(f"  Total: {len(files)}, Issues: {issues_count}, Unreadable: {unreadable}")
    
    if issues_count > 0:
        print(f"  ⚠ Files with issues:")
        for r in results:
            if r["issues"]:
                print(f"    {r['filename']}: {'; '.join(r['issues'])}")
    
    return results

print("=" * 60)
print("VALIDATING ALL SOURCE FILES")
print("=" * 60)
print()

val_positive   = validate_file_set(positive_files,   "positive")
print()
val_silence    = validate_file_set(silence_files,     "negative_silence")
print()
val_background = validate_file_set(background_files,  "negative_background")
print()

# Speech Commands: validate a random sample (full set is ~105K)
print("Speech commands: validating random sample of 500 for efficiency...")
rng_val = np.random.RandomState(SEED)
sc_sample_idx = rng_val.choice(
    len(speech_cmd_files),
    size=min(500, len(speech_cmd_files)),
    replace=False,
)
sc_sample = [speech_cmd_files[i] for i in sc_sample_idx]
val_sc_sample = validate_file_set(sc_sample, "speech_commands (sample)")

all_val_results = val_positive + val_silence + val_background + val_sc_sample
total_issues    = sum(1 for r in all_val_results if r["issues"])
total_unreadable = sum(1 for r in all_val_results if not r["readable"])

print()
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"  Total validated:  {len(all_val_results)}")
print(f"  Files with issues: {total_issues}")
print(f"  Unreadable files:  {total_unreadable}")
if total_issues == 0:
    print("  ✓ All validated files pass checks.")
else:
    print("  ⚠ Review the issues above before proceeding.")

VALIDATING ALL SOURCE FILES



Validating positive: 100%|██████████| 704/704 [00:03<00:00, 206.53it/s]


  Total: 704, Issues: 702, Unreadable: 0
  ⚠ Files with issues:
    ananya_001.wav: Duration=2.325s (expected 1.0s)
    ananya_002.wav: Duration=2.325s (expected 1.0s)
    ananya_003.wav: Duration=4.032s (expected 1.0s)
    ananya_004.wav: Duration=4.032s (expected 1.0s)
    ananya_005.wav: Duration=1.664s (expected 1.0s)
    ananya_006.wav: Duration=1.664s (expected 1.0s)
    ananya_007.wav: Duration=1.237s (expected 1.0s)
    ananya_008.wav: Duration=1.237s (expected 1.0s)
    ananya_009.wav: Duration=2.496s (expected 1.0s)
    ananya_010.wav: Duration=2.496s (expected 1.0s)
    ananya_011.wav: Duration=1.664s (expected 1.0s)
    ananya_012.wav: Duration=1.664s (expected 1.0s)
    ananya_013.wav: Duration=1.280s (expected 1.0s)
    ananya_014.wav: Duration=1.280s (expected 1.0s)
    ananya_015.wav: Duration=1.643s (expected 1.0s)
    ananya_016.wav: Duration=1.643s (expected 1.0s)
    ananya_017.wav: Duration=1.557s (expected 1.0s)
    ananya_018.wav: Duration=1.557s (expected 1.0s)


Validating negative_silence: 100%|██████████| 307/307 [00:01<00:00, 210.15it/s]


  Total: 307, Issues: 0, Unreadable: 0



Validating negative_background: 100%|██████████| 791/791 [00:03<00:00, 227.61it/s]


  Total: 791, Issues: 0, Unreadable: 0

Speech commands: validating random sample of 500 for efficiency...


Validating speech_commands (sample): 100%|██████████| 500/500 [00:01<00:00, 390.23it/s]

  Total: 500, Issues: 49, Unreadable: 0
  ⚠ Files with issues:
    happy_ab5d7179_nohash_0.wav: Duration=0.768s (expected 1.0s)
    happy_3659fc1c_nohash_0.wav: Duration=0.697s (expected 1.0s)
    dog_4874a7e9_nohash_1.wav: Duration=0.683s (expected 1.0s)
    nine_d37e4bf1_nohash_0.wav: Duration=0.929s (expected 1.0s)
    happy_b10b0654_nohash_1.wav: Duration=0.853s (expected 1.0s)
    five_acb9db68_nohash_0.wav: Duration=0.640s (expected 1.0s)
    two_6f5b4d3d_nohash_0.wav: Duration=0.789s (expected 1.0s)
    five_ea0cf37f_nohash_0.wav: Duration=0.758s (expected 1.0s)
    six_bf8d5617_nohash_0.wav: Duration=0.853s (expected 1.0s)
    yes_f4cae173_nohash_0.wav: Duration=0.683s (expected 1.0s)
    five_4beff0c5_nohash_1.wav: Duration=0.725s (expected 1.0s)
    up_7de97453_nohash_1.wav: Duration=0.836s (expected 1.0s)
    five_d3f22f0e_nohash_1.wav: Duration=0.939s (expected 1.0s)
    stop_151bfb79_nohash_0.wav: Duration=0.929s (expected 1.0s)
    nine_c678e0b3_nohash_0.wav: Duration=0.6

## 5. Speech Commands — Category Discovery

In [26]:
# ============================================================
# SECTION 5 — SPEECH COMMANDS CATEGORY DISCOVERY
# ============================================================

def extract_sc_category(filepath):
    """Extract word/category from Speech Commands filename.
    Format: <word>_<speakerhash>_nohash_<idx>.wav
    Category is everything before the first 8-char hex hash.
    """
    parts = filepath.stem.split('_')
    category_parts = []
    for p in parts:
        if len(p) >= 8 and all(c in '0123456789abcdef' for c in p):
            break
        category_parts.append(p)
    return '_'.join(category_parts) if category_parts else parts[0]

def extract_sc_speaker(filepath):
    """Extract speaker hash from Speech Commands filename."""
    parts = filepath.stem.split('_')
    for p in parts:
        if len(p) == 8 and all(c in '0123456789abcdef' for c in p):
            return p
    return "unknown"

# Build category → files mapping
sc_categories = defaultdict(list)
for f in speech_cmd_files:
    cat = extract_sc_category(f)
    sc_categories[cat].append(f)
sc_categories = dict(sorted(sc_categories.items()))

print("=" * 60)
print("SPEECH COMMANDS — CATEGORIES")
print("=" * 60)
print(f"Total categories discovered: {len(sc_categories)}")
print(f"Total files: {len(speech_cmd_files):,}")
print()
print(f"{'Category':<20s} {'Count':>8s}")
print("─" * 30)
for cat, files in sc_categories.items():
    print(f"{cat:<20s} {len(files):>8,}")
print("─" * 30)
print(f"{'TOTAL':<20s} {sum(len(v) for v in sc_categories.values()):>8,}")

SPEECH COMMANDS — CATEGORIES
Total categories discovered: 35
Total files: 105,829

Category                Count
──────────────────────────────
backward                1,664
bed                     2,014
bird                    2,064
cat                     2,031
dog                     2,128
down                    3,917
eight                   3,787
five                    4,052
follow                  1,579
forward                 1,557
four                    3,728
go                      3,880
happy                   2,054
house                   2,113
learn                   1,575
left                    3,801
marvin                  2,100
nine                    3,934
no                      3,941
off                     3,745
on                      3,845
one                     3,890
right                   3,778
seven                   3,998
sheila                  2,022
six                     3,860
stop                    3,872
three                   3,727
tree            

## 6. Speech Commands — Stratified Sampling (~100/category)

In [27]:
# ============================================================
# SECTION 6 — SPEECH COMMANDS STRATIFIED SAMPLING
# ============================================================

rng_sc = np.random.RandomState(SEED)

selected_sc = {}   # category → list of files
sc_selection_report = []

print("=" * 60)
print("SPEECH COMMANDS — STRATIFIED SAMPLING")
print("=" * 60)
print(f"Target per category: {TARGET_PER_SC_CATEGORY}")
print()
print(f"{'Category':<20s} {'Available':>10s} {'Selected':>10s} {'Note'}")
print("─" * 65)

total_selected = 0
for cat, files in sc_categories.items():
    n_available = len(files)
    n_select = min(TARGET_PER_SC_CATEGORY, n_available)
    
    if n_available <= TARGET_PER_SC_CATEGORY:
        chosen = files[:]
        note = "← ALL (fewer than target)"
    else:
        indices = rng_sc.choice(n_available, size=n_select, replace=False)
        chosen = [files[i] for i in sorted(indices)]
        note = ""
    
    selected_sc[cat] = chosen
    total_selected += len(chosen)
    sc_selection_report.append({
        "category": cat,
        "available": n_available,
        "selected": len(chosen),
    })
    print(f"{cat:<20s} {n_available:>10,} {len(chosen):>10,} {note}")

print("─" * 65)
print(f"{'TOTAL':<20s} {sum(len(v) for v in sc_categories.values()):>10,} {total_selected:>10,}")

# Flatten
selected_sc_files = []
for cat, files in selected_sc.items():
    for f in files:
        selected_sc_files.append((f, cat))

print(f"\nTotal Speech Commands files selected: {len(selected_sc_files)}")

# Save selection list for reproducibility
sel_rows = []
for f, cat in selected_sc_files:
    sel_rows.append({
        "filepath": str(f),
        "filename": f.name,
        "category": cat,
        "speaker": extract_sc_speaker(f),
    })
sel_df = pd.DataFrame(sel_rows)
sel_path = V2_DIRS["manifests"] / "speech_commands_selection.csv"
sel_df.to_csv(sel_path, index=False)
print(f"Saved selection list: {sel_path}")

SPEECH COMMANDS — STRATIFIED SAMPLING
Target per category: 100

Category              Available   Selected Note
─────────────────────────────────────────────────────────────────
backward                  1,664        100 
bed                       2,014        100 
bird                      2,064        100 
cat                       2,031        100 
dog                       2,128        100 
down                      3,917        100 
eight                     3,787        100 
five                      4,052        100 
follow                    1,579        100 
forward                   1,557        100 
four                      3,728        100 
go                        3,880        100 
happy                     2,054        100 
house                     2,113        100 
learn                     1,575        100 
left                      3,801        100 
marvin                    2,100        100 
nine                      3,934        100 
no                        3,94

## 7. Positive Split — Per-Speaker 70/15/15

**Every speaker appears in train, validation, AND test.**  
Each speaker's recordings are independently split 70/15/15.  
This preserves speaker diversity across all splits.

In [28]:
# ============================================================
# SECTION 7 — POSITIVE SPLIT (PER-SPEAKER 70/15/15)
# ============================================================

positive_records = []

print("=" * 60)
print("POSITIVE SPLIT — PER-SPEAKER")
print("=" * 60)
print()
print(f"{'Speaker':<12s} {'Total':>6s} {'Train':>6s} {'Val':>6s} {'Test':>6s}  "
      f"{'Train%':>7s} {'Val%':>7s} {'Test%':>7s}")
print("─" * 75)

rng_pos = np.random.RandomState(SEED)

for speaker in sorted(pos_speakers.keys()):
    files = pos_speakers[speaker]
    n = len(files)
    
    # Shuffle deterministically
    indices = list(range(n))
    rng_pos.shuffle(indices)
    
    n_train = int(round(n * TRAIN_RATIO))
    n_val   = int(round(n * VAL_RATIO))
    n_test  = n - n_train - n_val
    
    # Ensure at least 1 in each split
    if n_test < 1 and n >= 3:
        n_test = 1
        n_train = n - n_val - n_test
    if n_val < 1 and n >= 3:
        n_val = 1
        n_train = n - n_val - n_test
    
    train_idx = indices[:n_train]
    val_idx   = indices[n_train:n_train + n_val]
    test_idx  = indices[n_train + n_val:]
    
    for i in train_idx:
        positive_records.append({
            "filepath": files[i],
            "speaker": speaker,
            "split": "train",
        })
    for i in val_idx:
        positive_records.append({
            "filepath": files[i],
            "speaker": speaker,
            "split": "validation",
        })
    for i in test_idx:
        positive_records.append({
            "filepath": files[i],
            "speaker": speaker,
            "split": "test",
        })
    
    pct_tr = n_train / n * 100
    pct_va = n_val / n * 100
    pct_te = n_test / n * 100
    print(f"{speaker:<12s} {n:>6d} {n_train:>6d} {n_val:>6d} {n_test:>6d}  "
          f"{pct_tr:>6.1f}% {pct_va:>6.1f}% {pct_te:>6.1f}%")

totals = Counter(r["split"] for r in positive_records)
print("─" * 75)
print(f"{'TOTAL':<12s} {len(positive_records):>6d} {totals['train']:>6d} "
      f"{totals['validation']:>6d} {totals['test']:>6d}")
print()
print("✓ Every speaker appears in train, validation, AND test.")

POSITIVE SPLIT — PER-SPEAKER

Speaker       Total  Train    Val   Test   Train%    Val%   Test%
───────────────────────────────────────────────────────────────────────────
ananya           78     55     12     11    70.5%   15.4%   14.1%
ark              98     69     15     14    70.4%   15.3%   14.3%
ishita           78     55     12     11    70.5%   15.4%   14.1%
mayank           96     67     14     15    69.8%   14.6%   15.6%
umang           100     70     15     15    70.0%   15.0%   15.0%
vitthal         254    178     38     38    70.1%   15.0%   15.0%
───────────────────────────────────────────────────────────────────────────
TOTAL           704    494    106    104

✓ Every speaker appears in train, validation, AND test.


## 8. Silence & Background — Source-Aware Split

**Critical leakage prevention:** clips from the same original recording
must ALL go to the same split. We split by *source recording*, not by
individual clip.

### Silence
All silence clips come from a single source recording (`Recording (53)`).
Source-aware splitting means **all 307 clips must go to one split**.
We put them ALL in **train** since we cannot split a single source.
This is reported transparently.

### Background
6 distinct source recordings. We assign entire sources to splits.

In [29]:
# ============================================================
# SECTION 8 — SILENCE SOURCE-AWARE SPLIT
# ============================================================

print("=" * 60)
print("SILENCE — SOURCE-AWARE SPLIT")
print("=" * 60)

silence_records = []
n_silence_sources = len(silence_sources)

print(f"\nSilence source recordings found: {n_silence_sources}")
for src, clips in sorted(silence_sources.items()):
    print(f"  {src!r}: {len(clips)} clips")

if n_silence_sources == 1:
    # ALL clips from one source → must all go to same split
    sole_source = list(silence_sources.keys())[0]
    print(f"\n⚠ LIMITATION: All {len(silence_files)} silence clips come from ONE source "
          f"recording ({sole_source!r}).")
    print("  Source-aware splitting requires all clips to stay together.")
    print("  → Assigning ALL silence clips to TRAIN.")
    print("  → Validation and test will have 0 silence clips.")
    print("  → This is reported transparently; a separate unseen-silence")
    print("    evaluation should be conducted when new recordings are available.")
    
    for f in silence_files:
        silence_records.append({
            "filepath": f,
            "source_recording_id": sole_source,
            "split": "train",
        })
else:
    # Multiple sources — split source-level
    source_names = sorted(silence_sources.keys())
    rng_sil = np.random.RandomState(SEED)
    rng_sil.shuffle(source_names)
    
    n_src = len(source_names)
    n_train_src = max(1, int(round(n_src * TRAIN_RATIO)))
    n_val_src   = max(1, int(round(n_src * VAL_RATIO)))
    n_test_src  = n_src - n_train_src - n_val_src
    if n_test_src < 1:
        n_test_src = 1
        n_train_src = n_src - n_val_src - n_test_src
    
    train_srcs = source_names[:n_train_src]
    val_srcs   = source_names[n_train_src:n_train_src + n_val_src]
    test_srcs  = source_names[n_train_src + n_val_src:]
    
    for src in train_srcs:
        for f in silence_sources[src]:
            silence_records.append({"filepath": f, "source_recording_id": src, "split": "train"})
    for src in val_srcs:
        for f in silence_sources[src]:
            silence_records.append({"filepath": f, "source_recording_id": src, "split": "validation"})
    for src in test_srcs:
        for f in silence_sources[src]:
            silence_records.append({"filepath": f, "source_recording_id": src, "split": "test"})

sil_split_counts = Counter(r["split"] for r in silence_records)
print(f"\nSilence split counts:")
for s in SPLITS:
    print(f"  {s:12s}: {sil_split_counts.get(s, 0)}")

SILENCE — SOURCE-AWARE SPLIT

Silence source recordings found: 1
  'Recording (53)': 307 clips

⚠ LIMITATION: All 307 silence clips come from ONE source recording ('Recording (53)').
  Source-aware splitting requires all clips to stay together.
  → Assigning ALL silence clips to TRAIN.
  → Validation and test will have 0 silence clips.
  → This is reported transparently; a separate unseen-silence
    evaluation should be conducted when new recordings are available.

Silence split counts:
  train       : 307
  validation  : 0
  test        : 0


In [30]:
# ============================================================
# BACKGROUND — SOURCE-AWARE SPLIT
# ============================================================

print("=" * 60)
print("BACKGROUND — SOURCE-AWARE SPLIT")
print("=" * 60)

bg_records = []
n_bg_sources = len(bg_sources)
source_names_bg = sorted(bg_sources.keys())

print(f"\nBackground source recordings: {n_bg_sources}")
for src in source_names_bg:
    print(f"  {src}: {len(bg_sources[src])} clips")

if n_bg_sources < 3:
    print(f"\n⚠ Only {n_bg_sources} background sources — cannot split into 3 groups.")
    print("  Assigning all to TRAIN.")
    for src in source_names_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "train"})
else:
    # Assign sources to splits proportionally
    rng_bg = np.random.RandomState(SEED)
    rng_bg.shuffle(source_names_bg)
    
    n_src = len(source_names_bg)
    n_train_src = max(1, int(round(n_src * TRAIN_RATIO)))
    n_val_src   = max(1, int(round(n_src * VAL_RATIO)))
    n_test_src  = n_src - n_train_src - n_val_src
    if n_test_src < 1:
        n_test_src = 1
        n_train_src = n_src - n_val_src - n_test_src
    
    train_bg = source_names_bg[:n_train_src]
    val_bg   = source_names_bg[n_train_src:n_train_src + n_val_src]
    test_bg  = source_names_bg[n_train_src + n_val_src:]
    
    print(f"\nSource assignment:")
    print(f"  Train sources ({len(train_bg)}): {train_bg}")
    print(f"  Val sources   ({len(val_bg)}):   {val_bg}")
    print(f"  Test sources  ({len(test_bg)}):  {test_bg}")
    
    for src in train_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "train"})
    for src in val_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "validation"})
    for src in test_bg:
        for f in bg_sources[src]:
            bg_records.append({"filepath": f, "source_recording_id": src, "split": "test"})

bg_split_counts = Counter(r["split"] for r in bg_records)
print(f"\nBackground split counts:")
for s in SPLITS:
    print(f"  {s:12s}: {bg_split_counts.get(s, 0)}")
total_bg = sum(bg_split_counts.values())
print(f"  {'TOTAL':12s}: {total_bg}")
for s in SPLITS:
    pct = bg_split_counts.get(s, 0) / total_bg * 100 if total_bg else 0
    print(f"  {s} ratio: {pct:.1f}%")
print()
print("Note: Ratios may differ from 70/15/15 because source integrity")
print("takes priority over exact percentages.")

BACKGROUND — SOURCE-AWARE SPLIT

Background source recordings: 6
  background_doing_the_dishes: 189 clips
  background_dude_miaowing: 122 clips
  background_exercise_bike: 121 clips
  background_pink_noise: 119 clips
  background_running_tap: 121 clips
  background_white_noise: 119 clips

Source assignment:
  Train sources (4): ['background_doing_the_dishes', 'background_dude_miaowing', 'background_white_noise', 'background_exercise_bike']
  Val sources   (1):   ['background_running_tap']
  Test sources  (1):  ['background_pink_noise']

Background split counts:
  train       : 551
  validation  : 121
  test        : 119
  TOTAL       : 791
  train ratio: 69.7%
  validation ratio: 15.3%
  test ratio: 15.0%

Note: Ratios may differ from 70/15/15 because source integrity
takes priority over exact percentages.


In [31]:
# ============================================================
# SPEECH COMMANDS — STRATIFIED SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

sc_records = []

# Build flat list with subcategory labels for stratification
sc_flat_files = []
sc_flat_labels = []
for cat, files in selected_sc.items():
    for f in files:
        sc_flat_files.append(f)
        sc_flat_labels.append(cat)

try:
    # Two-stage stratified split
    train_files, temp_files, train_labels, temp_labels = train_test_split(
        sc_flat_files, sc_flat_labels,
        test_size=(VAL_RATIO + TEST_RATIO),
        stratify=sc_flat_labels,
        random_state=SEED,
    )
    relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_files, test_files, val_labels, test_labels = train_test_split(
        temp_files, temp_labels,
        test_size=relative_test,
        stratify=temp_labels,
        random_state=SEED,
    )
    sc_split_method = "stratified-by-word"
except ValueError:
    # Fallback if stratification fails
    rng_sc_split = np.random.RandomState(SEED)
    combined = list(zip(sc_flat_files, sc_flat_labels))
    rng_sc_split.shuffle(combined)
    n = len(combined)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    train_files = [c[0] for c in combined[:n_train]]
    train_labels = [c[1] for c in combined[:n_train]]
    val_files = [c[0] for c in combined[n_train:n_train + n_val]]
    val_labels = [c[1] for c in combined[n_train:n_train + n_val]]
    test_files = [c[0] for c in combined[n_train + n_val:]]
    test_labels = [c[1] for c in combined[n_train + n_val:]]
    sc_split_method = "random-fallback"

for f, lab in zip(train_files, train_labels):
    sc_records.append({"filepath": f, "subcategory": lab, "split": "train"})
for f, lab in zip(val_files, val_labels):
    sc_records.append({"filepath": f, "subcategory": lab, "split": "validation"})
for f, lab in zip(test_files, test_labels):
    sc_records.append({"filepath": f, "subcategory": lab, "split": "test"})

print("=" * 60)
print("SPEECH COMMANDS — SPLIT")
print("=" * 60)
print(f"Method: {sc_split_method}")
sc_split_counts = Counter(r["split"] for r in sc_records)
for s in SPLITS:
    print(f"  {s:12s}: {sc_split_counts.get(s, 0)}")

# Verify category representation across splits
sc_df_check = pd.DataFrame(sc_records)
cats_per_split = {}
for s in SPLITS:
    cats_per_split[s] = set(sc_df_check[sc_df_check["split"] == s]["subcategory"])
all_sc_cats = set(sc_flat_labels)
for s in SPLITS:
    missing = all_sc_cats - cats_per_split[s]
    if missing:
        print(f"  ⚠ Categories missing from {s}: {missing}")
    else:
        print(f"  ✓ All {len(all_sc_cats)} categories represented in {s}")

SPEECH COMMANDS — SPLIT
Method: stratified-by-word
  train       : 2450
  validation  : 525
  test        : 525
  ✓ All 35 categories represented in train
  ✓ All 35 categories represented in validation
  ✓ All 35 categories represented in test


## 9. Dataset Manifest Creation

Build the full manifest with all required fields including SHA-256 hashes.

In [32]:
# ============================================================
# SECTION 9 — BUILD FULL MANIFEST
# ============================================================

def file_sha256(filepath):
    """Compute SHA-256 hash of a file."""
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def get_wav_info(filepath):
    """Get sample_rate, channels, duration from a WAV file."""
    try:
        with wave.open(str(filepath), 'rb') as wf:
            sr = wf.getframerate()
            ch = wf.getnchannels()
            dur = wf.getnframes() / sr
            return sr, ch, dur
    except Exception:
        return None, None, None

print("Building manifest (this includes SHA-256 hashing and may take a few minutes)...")
print()

manifest_rows = []

# ── Positive ──
print("Hashing positive files...")
for rec in tqdm(positive_records, desc="Positive"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_POSITIVE,
        "category": "positive",
        "subcategory": "vaani",
        "speaker_or_source_id": rec["speaker"],
        "source_recording_id": rec["speaker"],  # each speaker is their own source
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

# ── Silence ──
print("Hashing silence files...")
for rec in tqdm(silence_records, desc="Silence"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_silence",
        "subcategory": "silence",
        "speaker_or_source_id": rec["source_recording_id"],
        "source_recording_id": rec["source_recording_id"],
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

# ── Background ──
print("Hashing background files...")
for rec in tqdm(bg_records, desc="Background"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_background",
        "subcategory": rec["source_recording_id"],
        "speaker_or_source_id": rec["source_recording_id"],
        "source_recording_id": rec["source_recording_id"],
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

# ── Speech Commands ──
print("Hashing speech command files...")
for rec in tqdm(sc_records, desc="Speech Cmds"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_speech_commands",
        "subcategory": rec["subcategory"],
        "speaker_or_source_id": extract_sc_speaker(f),
        "source_recording_id": f.name,  # each SC file is independent
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

manifest_df = pd.DataFrame(manifest_rows)

print(f"\nManifest built: {len(manifest_df)} rows")
print(f"Columns: {list(manifest_df.columns)}")

Building manifest (this includes SHA-256 hashing and may take a few minutes)...

Hashing positive files...


Positive: 100%|██████████| 704/704 [00:00<00:00, 1559.79it/s]


Hashing silence files...


Silence: 100%|██████████| 307/307 [00:00<00:00, 2165.01it/s]


Hashing background files...


Background: 100%|██████████| 791/791 [00:00<00:00, 2092.01it/s]


Hashing speech command files...


Speech Cmds: 100%|██████████| 3500/3500 [00:02<00:00, 1327.14it/s]


Manifest built: 5302 rows
Columns: ['source_path', 'relative_source_path', 'filename', 'label', 'category', 'subcategory', 'speaker_or_source_id', 'source_recording_id', 'split', 'sha256', 'sample_rate', 'channels', 'duration']


In [33]:
# ============================================================
# DEDUPLICATION — Remove byte-identical files across splits
# ============================================================
# Some files in the source dataset (especially Speech Commands)
# have identical audio content despite different filenames.
# If these end up in different splits, it causes data leakage.
# We deduplicate by SHA-256 hash BEFORE copying.

print("=" * 60)
print("DEDUPLICATION (SHA-256)")
print("=" * 60)

hash_counts = manifest_df["sha256"].value_counts()
dupe_hashes = set(hash_counts[hash_counts > 1].index)

if not dupe_hashes:
    print("\nNo duplicate hashes found — no deduplication needed.")
else:
    print(f"\nFound {len(dupe_hashes)} SHA-256 values with multiple files")

    # Save per-category counts before dedup
    cat_counts_before = manifest_df.groupby("category").size().to_dict()
    split_counts_before = manifest_df.groupby("split").size().to_dict()

    rows_to_drop = []
    cross_split_count = 0
    same_split_count = 0

    for h in sorted(dupe_hashes):
        dupe_rows = manifest_df[manifest_df["sha256"] == h]
        splits_in_group = set(dupe_rows["split"])

        if len(splits_in_group) > 1:
            cross_split_count += 1
        else:
            same_split_count += 1

        # Keep only the FIRST occurrence, drop all others
        indices_to_drop = dupe_rows.index[1:].tolist()
        rows_to_drop.extend(indices_to_drop)

    n_before = len(manifest_df)
    manifest_df = manifest_df.drop(index=rows_to_drop).reset_index(drop=True)
    n_after = len(manifest_df)
    n_removed = n_before - n_after

    print(f"  Cross-split duplicate groups: {cross_split_count}")
    print(f"  Same-split duplicate groups:  {same_split_count}")
    print(f"  Total duplicate groups:       {len(dupe_hashes)}")
    print(f"  Rows before dedup:            {n_before}")
    print(f"  Rows removed:                 {n_removed}")
    print(f"  Rows after dedup:             {n_after}")

    # Per-category breakdown
    print()
    print("  Removed files by category:")
    cat_counts_after = manifest_df.groupby("category").size().to_dict()
    for cat in ["positive", "negative_silence", "negative_background", "negative_speech_commands"]:
        before = cat_counts_before.get(cat, 0)
        after = cat_counts_after.get(cat, 0)
        removed = before - after
        if removed > 0:
            print(f"    {cat}: {removed} removed ({before} -> {after})")

    # Per-split breakdown
    print()
    print("  Remaining files per split:")
    for s in ["train", "validation", "test"]:
        sdf = manifest_df[manifest_df["split"] == s]
        n_pos = len(sdf[sdf["label"] == 1])
        n_neg = len(sdf[sdf["label"] == 0])
        print(f"    {s:12s}: {len(sdf):>5d} (pos: {n_pos}, neg: {n_neg})")

    # Verify no cross-split hash duplicates remain
    for s1, s2 in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        h1 = set(manifest_df[manifest_df["split"] == s1]["sha256"])
        h2 = set(manifest_df[manifest_df["split"] == s2]["sha256"])
        overlap = h1 & h2
        if overlap:
            print(f"\n  ✗ {len(overlap)} hash duplicates still between {s1} and {s2}")
        else:
            print(f"  ✓ No hash overlap between {s1} and {s2}")

    # Update the split assignment dictionaries so downstream copy uses deduped list
    # Rebuild from manifest_df
    positive_splits = {"train": [], "validation": [], "test": []}
    silence_splits = {"train": [], "validation": [], "test": []}
    background_splits = {"train": [], "validation": [], "test": []}
    sc_splits = {"train": [], "validation": [], "test": []}

    for _, row in manifest_df.iterrows():
        p = Path(row["source_path"])
        s = row["split"]
        cat = row["category"]
        if cat == "positive":
            positive_splits[s].append(p)
        elif cat == "negative_silence":
            silence_splits[s].append(p)
        elif cat == "negative_background":
            background_splits[s].append(p)
        elif cat == "negative_speech_commands":
            sc_splits[s].append(p)

    print(f"\n  ✓ Split dictionaries rebuilt from deduped manifest")

DEDUPLICATION (SHA-256)

Found 356 SHA-256 values with multiple files
  Cross-split duplicate groups: 167
  Same-split duplicate groups:  189
  Total duplicate groups:       356
  Rows before dedup:            5302
  Rows removed:                 356
  Rows after dedup:             4946

  Removed files by category:
    positive: 352 removed (704 -> 352)
    negative_speech_commands: 4 removed (3500 -> 3496)

  Remaining files per split:
    train       :  3629 (pos: 322, neg: 3307)
    validation  :   668 (pos: 24, neg: 644)
    test        :   649 (pos: 6, neg: 643)
  ✓ No hash overlap between train and validation
  ✓ No hash overlap between train and test
  ✓ No hash overlap between validation and test

  ✓ Split dictionaries rebuilt from deduped manifest


## 10. Copy Dataset into model_v2/data/

Only copies files — never modifies `dataset/raw/`.

In [34]:
# ============================================================
# SECTION 10 — COPY FILES INTO model_v2/data/
# ============================================================

print("=" * 60)
print("COPYING FILES TO model_v2/data/")
print("=" * 60)

copy_count = 0
copy_errors = []

for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Copying"):
    src = Path(row["source_path"])
    dst_dir = V2_DIRS["data"] / row["split"] / row["category"]
    dst = dst_dir / row["filename"]
    
    try:
        if not dst.exists():
            shutil.copy2(str(src), str(dst))
        copy_count += 1
    except Exception as e:
        copy_errors.append(f"{src.name}: {e}")

print(f"\nCopied/verified: {copy_count} files")
if copy_errors:
    print(f"Errors: {len(copy_errors)}")
    for err in copy_errors[:10]:
        print(f"  {err}")
else:
    print("✓ No copy errors.")

# Verify file counts match
print()
print("Verification — files in model_v2/data/:")
for split in SPLITS:
    for cat in CATEGORIES:
        d = V2_DIRS["data"] / split / cat
        n = len(list(d.glob("*.wav")))
        expected = len(manifest_df[
            (manifest_df["split"] == split) & (manifest_df["category"] == cat)
        ])
        status = "✓" if n == expected else "✗"
        print(f"  {status} {split:12s} / {cat:30s}: {n:>5d} (expected {expected})")

COPYING FILES TO model_v2/data/


Copying: 100%|██████████| 4946/4946 [00:06<00:00, 787.26it/s]


Copied/verified: 4946 files
✓ No copy errors.

Verification — files in model_v2/data/:
  ✓ train        / positive                      :   322 (expected 322)
  ✓ train        / negative_silence              :   307 (expected 307)
  ✓ train        / negative_background           :   551 (expected 551)
  ✓ train        / negative_speech_commands      :  2449 (expected 2449)
  ✓ validation   / positive                      :    24 (expected 24)
  ✓ validation   / negative_silence              :     0 (expected 0)
  ✓ validation   / negative_background           :   121 (expected 121)
  ✓ validation   / negative_speech_commands      :   523 (expected 523)
  ✓ test         / positive                      :     6 (expected 6)
  ✓ test         / negative_silence              :     0 (expected 0)
  ✓ test         / negative_background           :   119 (expected 119)
  ✓ test         / negative_speech_commands      :   524 (expected 524)


In [35]:
# ── Save manifest ────────────────────────────────────────────

csv_path = V2_DIRS["manifests"] / "v2_dataset_manifest.csv"
manifest_df.to_csv(csv_path, index=False)
print(f"Saved manifest CSV:  {csv_path}")
print(f"  Rows: {len(manifest_df)}")

json_path = V2_DIRS["manifests"] / "v2_dataset_manifest.json"
manifest_df.to_json(json_path, orient="records", indent=2)
print(f"Saved manifest JSON: {json_path}")

Saved manifest CSV:  c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data\manifests\v2_dataset_manifest.csv
  Rows: 4946
Saved manifest JSON: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data\manifests\v2_dataset_manifest.json


## 11. Leakage Verification

Multi-level leakage checks:
1. **Hash-based**: No identical SHA-256 hash across splits
2. **Filename-based**: No identical filename across splits within same category
3. **Source-recording-based**: No source recording ID appears in multiple splits
4. **Speaker-based (positive)**: Verify speaker coverage is intentional

In [36]:
# ============================================================
# SECTION 11 — LEAKAGE VERIFICATION
# ============================================================

print("=" * 60)
print("LEAKAGE VERIFICATION")
print("=" * 60)

leakage_found = False

# ── Check 1: SHA-256 hash uniqueness across splits ──────────
print("\n── Check 1: SHA-256 hash — no identical file across splits ──")

for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i+1:]:
            hashes_s1 = set(cat_df[cat_df["split"] == s1]["sha256"])
            hashes_s2 = set(cat_df[cat_df["split"] == s2]["sha256"])
            overlap = hashes_s1 & hashes_s2
            if overlap:
                print(f"  ✗ HASH LEAKAGE in {cat}: {len(overlap)} identical files "
                      f"in {s1} and {s2}")
                leakage_found = True
            else:
                n1 = len(hashes_s1)
                n2 = len(hashes_s2)
                if n1 > 0 and n2 > 0:
                    print(f"  ✓ {cat}: no hash overlap between {s1}({n1}) and {s2}({n2})")
                elif n1 == 0 or n2 == 0:
                    print(f"  ○ {cat}: {s1}({n1}) / {s2}({n2}) — one split empty (by design)")

# ── Check 2: Filename uniqueness within category ────────────
print("\n── Check 2: Filename uniqueness within each category ──")

for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i+1:]:
            fns1 = set(cat_df[cat_df["split"] == s1]["filename"])
            fns2 = set(cat_df[cat_df["split"] == s2]["filename"])
            overlap = fns1 & fns2
            if overlap:
                print(f"  ✗ FILENAME LEAKAGE in {cat}: {len(overlap)} files in "
                      f"both {s1} and {s2}")
                leakage_found = True
            else:
                if len(fns1) > 0 and len(fns2) > 0:
                    print(f"  ✓ {cat}: no filename overlap between {s1} and {s2}")

# ── Check 3: Source recording integrity ──────────────────────
print("\n── Check 3: Source recording — no source in multiple splits ──")

for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    source_splits = cat_df.groupby("source_recording_id")["split"].apply(set)
    for src_id, splits_set in source_splits.items():
        if len(splits_set) > 1:
            print(f"  ✗ SOURCE LEAKAGE in {cat}: source {src_id!r} appears in "
                  f"{splits_set}")
            leakage_found = True
        else:
            pass  # suppress per-source OK messages for brevity
    
    n_sources = len(source_splits)
    n_leaked = sum(1 for s in source_splits if len(s) > 1)
    print(f"  {'✗' if n_leaked else '✓'} {cat}: {n_sources} sources, "
          f"{n_leaked} with cross-split leakage")

# ── Check 4: Positive speaker coverage (informational) ──────
print("\n── Check 4: Positive speaker coverage ──")

pos_df = manifest_df[manifest_df["category"] == "positive"]
for spk in sorted(pos_df["speaker_or_source_id"].unique()):
    spk_splits = set(pos_df[pos_df["speaker_or_source_id"] == spk]["split"])
    if spk_splits == set(SPLITS):
        print(f"  ✓ {spk}: present in all 3 splits")
    else:
        print(f"  ⚠ {spk}: only in {spk_splits} (expected all 3)")

# ── Check 5: Global duplicate hashes ─────────────────────────
print("\n── Check 5: Global duplicate hashes ──")

hash_counts = manifest_df["sha256"].value_counts()
dupe_hashes = hash_counts[hash_counts > 1]
if len(dupe_hashes) > 0:
    print(f"  ⚠ {len(dupe_hashes)} SHA-256 values appear more than once")
    for h, cnt in dupe_hashes.head(5).items():
        dupe_rows = manifest_df[manifest_df["sha256"] == h][["filename", "category", "split"]]
        print(f"    Hash {h[:16]}...: {cnt} occurrences")
        for _, r in dupe_rows.iterrows():
            print(f"      {r['filename']} | {r['category']} | {r['split']}")
else:
    print(f"  ✓ All {len(manifest_df)} files have unique SHA-256 hashes")

# ── Summary ──────────────────────────────────────────────────
print()
if leakage_found:
    print("✗ LEAKAGE DETECTED — review the issues above.")
else:
    print("✓ NO LEAKAGE DETECTED — dataset splits are clean.")

LEAKAGE VERIFICATION

── Check 1: SHA-256 hash — no identical file across splits ──
  ✓ positive: no hash overlap between train(322) and validation(24)
  ✓ positive: no hash overlap between train(322) and test(6)
  ✓ positive: no hash overlap between validation(24) and test(6)
  ○ negative_silence: train(307) / validation(0) — one split empty (by design)
  ○ negative_silence: train(307) / test(0) — one split empty (by design)
  ○ negative_silence: validation(0) / test(0) — one split empty (by design)
  ✓ negative_background: no hash overlap between train(551) and validation(121)
  ✓ negative_background: no hash overlap between train(551) and test(119)
  ✓ negative_background: no hash overlap between validation(121) and test(119)
  ✓ negative_speech_commands: no hash overlap between train(2449) and validation(523)
  ✓ negative_speech_commands: no hash overlap between train(2449) and test(524)
  ✓ negative_speech_commands: no hash overlap between validation(523) and test(524)

── Check 2

## 12. Dataset Statistics

In [37]:
# ============================================================
# SECTION 12 — DATASET STATISTICS
# ============================================================

print("=" * 70)
print("  V2 DATASET SUMMARY")
print("=" * 70)

# ── Per-category per-split ───────────────────────────────────
print()
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    cat_label = {
        "positive": "Positive (Vaani)",
        "negative_silence": "Silence",
        "negative_background": "Background",
        "negative_speech_commands": "Speech Commands",
    }[cat]
    print(f"{cat_label}:")
    for s in SPLITS:
        n = len(cat_df[cat_df["split"] == s])
        print(f"  {s:12s}: {n:>5d}")
    print(f"  {'total':12s}: {len(cat_df):>5d}")
    print()

# ── Totals ───────────────────────────────────────────────────
print("Total:")
for s in SPLITS:
    sdf = manifest_df[manifest_df["split"] == s]
    n_pos = len(sdf[sdf["label"] == LABEL_POSITIVE])
    n_neg = len(sdf[sdf["label"] == LABEL_NEGATIVE])
    print(f"  {s:12s}: {len(sdf):>5d}  (pos: {n_pos:>4d}, neg: {n_neg:>4d})")
print(f"  {'GRAND TOTAL':12s}: {len(manifest_df):>5d}")

# ── Speakers ─────────────────────────────────────────────────
print()
print("─" * 70)
print("SPEAKERS")
print("─" * 70)
pos_df = manifest_df[manifest_df["category"] == "positive"]
print(f"Number of speakers: {pos_df['speaker_or_source_id'].nunique()}")
print()
for spk in sorted(pos_df["speaker_or_source_id"].unique()):
    spk_df = pos_df[pos_df["speaker_or_source_id"] == spk]
    spk_splits = {s: len(spk_df[spk_df["split"] == s]) for s in SPLITS}
    print(f"  {spk:<12s}: total={len(spk_df):>4d}  "
          f"train={spk_splits['train']:>3d}  "
          f"val={spk_splits['validation']:>3d}  "
          f"test={spk_splits['test']:>3d}")

# ── Speech Commands categories ───────────────────────────────
print()
print("─" * 70)
print("SPEECH COMMANDS CATEGORIES")
print("─" * 70)
sc_df = manifest_df[manifest_df["category"] == "negative_speech_commands"]
sc_cats_summary = sc_df.groupby("subcategory").size().sort_index()
print(f"Total categories: {len(sc_cats_summary)}")
for cat_name, cnt in sc_cats_summary.items():
    print(f"  {cat_name:<20s}: {cnt:>4d}")

# ── Source recordings per split ──────────────────────────────
print()
print("─" * 70)
print("SOURCE RECORDINGS PER SPLIT")
print("─" * 70)
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    cat_label = "Silence" if "silence" in cat else "Background"
    for s in SPLITS:
        sources = cat_df[cat_df["split"] == s]["source_recording_id"].unique()
        print(f"  {cat_label} {s:12s}: {len(sources)} source(s): {list(sources)}")

# ── Invalid / problematic files ──────────────────────────────
print()
print("─" * 70)
print("INVALID FILES")
print("─" * 70)
total_issues_found = sum(1 for r in all_val_results if r["issues"])
if total_issues_found == 0:
    print("  No invalid files found during validation.")
else:
    print(f"  {total_issues_found} files had validation issues (see Section 4 output)")

# ── Positive:Negative ratio ─────────────────────────────────
print()
print("─" * 70)
print("CLASS BALANCE")
print("─" * 70)
n_pos_total = len(manifest_df[manifest_df["label"] == LABEL_POSITIVE])
n_neg_total = len(manifest_df[manifest_df["label"] == LABEL_NEGATIVE])
ratio = n_neg_total / n_pos_total if n_pos_total > 0 else float('inf')
print(f"  Total positive: {n_pos_total}")
print(f"  Total negative: {n_neg_total}")
print(f"  Ratio positive:negative = 1:{ratio:.2f}")
print(f"  (Intentionally negative-heavy — class imbalance handled during training)")

print()
print(f"  Random seed: {SEED}")

  V2 DATASET SUMMARY

Positive (Vaani):
  train       :   322
  validation  :    24
  test        :     6
  total       :   352

Silence:
  train       :   307
  validation  :     0
  test        :     0
  total       :   307

Background:
  train       :   551
  validation  :   121
  test        :   119
  total       :   791

Speech Commands:
  train       :  2449
  validation  :   523
  test        :   524
  total       :  3496

Total:
  train       :  3629  (pos:  322, neg: 3307)
  validation  :   668  (pos:   24, neg:  644)
  test        :   649  (pos:    6, neg:  643)
  GRAND TOTAL :  4946

──────────────────────────────────────────────────────────────────────
SPEAKERS
──────────────────────────────────────────────────────────────────────
Number of speakers: 6

  ananya      : total=  39  train= 37  val=  1  test=  1
  ark         : total=  49  train= 43  val=  6  test=  0
  ishita      : total=  39  train= 37  val=  2  test=  0
  mayank      : total=  48  train= 43  val=  4  test=

## 13. Final Sanity Checks

In [38]:
# ============================================================
# SECTION 13 — FINAL SANITY CHECKS
# ============================================================

print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

checks_passed = 0
checks_total  = 0

# Check 1: All source files accounted for
checks_total += 1
expected_total = (
    len(positive_files) +
    len(silence_files) +
    len(background_files) +
    len(selected_sc_files)
)
actual_total = len(manifest_df)
if actual_total == expected_total:
    print(f"✓ Check 1: Total files match ({actual_total} == {expected_total})")
    checks_passed += 1
else:
    print(f"✗ Check 1: Total mismatch ({actual_total} != {expected_total})")

# Check 2: No file appears in multiple splits
checks_total += 1
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i+1:]:
            fns1 = set(cat_df[cat_df["split"] == s1]["filename"])
            fns2 = set(cat_df[cat_df["split"] == s2]["filename"])
            assert not (fns1 & fns2), f"Leakage in {cat}: {fns1 & fns2}"
checks_passed += 1
print(f"✓ Check 2: No filename leakage across splits")

# Check 3: Labels are binary
checks_total += 1
unique_labels = set(manifest_df["label"].unique())
assert unique_labels.issubset({0, 1}), f"Unexpected labels: {unique_labels}"
checks_passed += 1
print(f"✓ Check 3: Labels are binary {{0, 1}}")

# Check 4: Every positive speaker in all 3 splits
checks_total += 1
pos_df_check = manifest_df[manifest_df["category"] == "positive"]
all_speakers_in_all = True
for spk in pos_df_check["speaker_or_source_id"].unique():
    spk_splits = set(pos_df_check[pos_df_check["speaker_or_source_id"] == spk]["split"])
    if spk_splits != set(SPLITS):
        all_speakers_in_all = False
        print(f"  ⚠ Speaker {spk} only in {spk_splits}")
if all_speakers_in_all:
    checks_passed += 1
    print(f"✓ Check 4: Every speaker in all 3 splits")
else:
    print(f"✗ Check 4: Some speakers not in all splits")

# Check 5: Source integrity for silence/background
checks_total += 1
source_ok = True
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for src_id in cat_df["source_recording_id"].unique():
        src_splits = set(cat_df[cat_df["source_recording_id"] == src_id]["split"])
        if len(src_splits) > 1:
            source_ok = False
            print(f"  ✗ {cat} source {src_id!r} in multiple splits: {src_splits}")
if source_ok:
    checks_passed += 1
    print(f"✓ Check 5: Source recording integrity preserved")
else:
    print(f"✗ Check 5: Source recording integrity violated")

# Check 6: SC categories represented in all splits
checks_total += 1
sc_df_final = manifest_df[manifest_df["category"] == "negative_speech_commands"]
all_cats = set(sc_df_final["subcategory"].unique())
sc_repr_ok = True
for s in SPLITS:
    s_cats = set(sc_df_final[sc_df_final["split"] == s]["subcategory"].unique())
    if s_cats != all_cats:
        missing = all_cats - s_cats
        sc_repr_ok = False
        print(f"  ⚠ {s} missing SC categories: {missing}")
if sc_repr_ok:
    checks_passed += 1
    print(f"✓ Check 6: All SC categories in all splits")
else:
    print(f"✗ Check 6: Some SC categories missing from splits")

# Check 7: Copied files match manifest
checks_total += 1
files_ok = True
for split in SPLITS:
    for cat in CATEGORIES:
        d = V2_DIRS["data"] / split / cat
        n_actual = len(list(d.glob("*.wav")))
        n_expected = len(manifest_df[
            (manifest_df["split"] == split) & (manifest_df["category"] == cat)
        ])
        if n_actual != n_expected:
            files_ok = False
            print(f"  ✗ {split}/{cat}: {n_actual} files (expected {n_expected})")
if files_ok:
    checks_passed += 1
    print(f"✓ Check 7: Copied file counts match manifest")
else:
    print(f"✗ Check 7: File count mismatch")

# Check 8: No hash duplicates across splits
checks_total += 1
hash_dupe_across = False
for i, s1 in enumerate(SPLITS):
    for s2 in SPLITS[i+1:]:
        h1 = set(manifest_df[manifest_df["split"] == s1]["sha256"])
        h2 = set(manifest_df[manifest_df["split"] == s2]["sha256"])
        if h1 & h2:
            hash_dupe_across = True
            print(f"  ✗ {len(h1 & h2)} hash duplicates between {s1} and {s2}")
if not hash_dupe_across:
    checks_passed += 1
    print(f"✓ Check 8: No SHA-256 hash duplicates across splits")
else:
    print(f"✗ Check 8: Hash duplicates found across splits")

print(f"\n{'=' * 60}")
print(f"SANITY CHECKS: {checks_passed}/{checks_total} PASSED")
print(f"{'=' * 60}")

SANITY CHECKS
✗ Check 1: Total mismatch (4946 != 5302)
✓ Check 2: No filename leakage across splits
✓ Check 3: Labels are binary {0, 1}
  ⚠ Speaker ark only in {'train', 'validation'}
  ⚠ Speaker ishita only in {'train', 'validation'}
✗ Check 4: Some speakers not in all splits
✓ Check 5: Source recording integrity preserved
✓ Check 6: All SC categories in all splits
✓ Check 7: Copied file counts match manifest
✓ Check 8: No SHA-256 hash duplicates across splits

SANITY CHECKS: 6/8 PASSED


In [39]:
# ============================================================
# CLEANUP TEMP DIRECTORY
# ============================================================

if TEMP_DIR.exists():
    temp_files = list(TEMP_DIR.iterdir())
    if temp_files:
        print(f"Cleaning {len(temp_files)} temp files...")
        shutil.rmtree(str(TEMP_DIR))
        TEMP_DIR.mkdir(parents=True, exist_ok=True)
        print("Temp directory cleaned.")
    else:
        print("Temp directory is empty — nothing to clean.")
else:
    print("No temp directory found.")

# ============================================================
# FINAL MESSAGE
# ============================================================

print()
print("=" * 70)
print("  V2 DATASET PREPARATION COMPLETE")
print("=" * 70)
print()
print("Outputs:")
print(f"  Data:      {V2_DIRS['data']}")
print(f"  Manifest:  {csv_path}")
print(f"  Manifest:  {json_path}")
print()
print("Next step:")
print("  Run 02_train_dscnn_v2.ipynb for augmentation,")
print("  feature extraction, and model training.")
print()
print("DO NOT modify anything under dataset/raw/.")
print("DO NOT train a model in this notebook.")
print("=" * 70)

Temp directory is empty — nothing to clean.

  V2 DATASET PREPARATION COMPLETE

Outputs:
  Data:      c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data
  Manifest:  c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data\manifests\v2_dataset_manifest.csv
  Manifest:  c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data\manifests\v2_dataset_manifest.json

Next step:
  Run 02_train_dscnn_v2.ipynb for augmentation,
  feature extraction, and model training.

DO NOT modify anything under dataset/raw/.
DO NOT train a model in this notebook.
